In [6]:
import requests
import pandas as pd
import os # Biblioteca para interagir com o sistema de arquivos
import json # Para salvar o JSON de forma legível

# --- Parte 1: Carregar os dados da API (o mesmo código de antes) ---
url = 'https://raw.githubusercontent.com/ingridcristh/challenge2-data-science/main/TelecomX_Data.json'

data = None # Inicializa 'data' para garantir que estará disponível fora do try-except
try:
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json() # Armazena o JSON bruto
        df = pd.DataFrame(data)
        print("Dados da API carregados com sucesso para o DataFrame.")
    else:
        print(f"Erro ao carregar os dados da API. Código de status: {response.status_code}")
except requests.exceptions.RequestException as e:
    print(f"Ocorreu um erro de conexão ao tentar carregar da API: {e}")
except ValueError as e:
    print(f"Erro ao decodificar JSON da API: {e}")

# --- Parte 2: Salvar os dados brutos no caminho data/raw ---
if data is not None: # Verifica se os dados foram realmente carregados
    # Define o caminho da pasta de dados brutos
    # O '..' significa "subir um nível" a partir do diretório atual (que é 'notebooks/')
    # Então, de 'notebooks/', você sobe para 'telecom_churn_analysis/',
    # e de lá, desce para 'data/raw'.
    raw_data_dir = os.path.join('..', 'data', 'raw') # Caminho correto para sair de 'notebooks'

    # Define o nome do arquivo
    file_name = 'TelecomX_Raw_Data.json'
    # Constrói o caminho completo do arquivo
    file_path = os.path.join(raw_data_dir, file_name)

    # Cria a pasta 'data/raw' se ela não existir
    # O `exist_ok=True` evita erro se a pasta já existe
    os.makedirs(raw_data_dir, exist_ok=True)

    try:
        # Salva os dados brutos no formato JSON com indentação
        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=4)
        print(f"\nDados brutos salvos com sucesso em: {file_path}")

    except IOError as e:
        print(f"Erro ao salvar o arquivo: {e}")
else:
    print("\nNão foi possível salvar os dados brutos, pois não foram carregados da API.")

Dados da API carregados com sucesso para o DataFrame.

Dados brutos salvos com sucesso em: ..\data\raw\TelecomX_Raw_Data.json


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7267 entries, 0 to 7266
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   customerID  7267 non-null   object
 1   Churn       7267 non-null   object
 2   customer    7267 non-null   object
 3   phone       7267 non-null   object
 4   internet    7267 non-null   object
 5   account     7267 non-null   object
dtypes: object(6)
memory usage: 340.8+ KB


In [31]:
df.head()

,customerID,Churn,customer,phone,internet,account
0,0002-ORFBO,No,"{'gender': 'Female', 'SeniorCitizen': 0, 'Part...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'DSL', 'OnlineSecurity': '...","{'Contract': 'One year', 'PaperlessBilling': '..."
1,0003-MKNFE,No,"{'gender': 'Male', 'SeniorCitizen': 0, 'Partne...","{'PhoneService': 'Yes', 'MultipleLines': 'Yes'}","{'InternetService': 'DSL', 'OnlineSecurity': '...","{'Contract': 'Month-to-month', 'PaperlessBilli..."
2,0004-TLHLJ,Yes,"{'gender': 'Male', 'SeniorCitizen': 0, 'Partne...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'Fiber optic', 'OnlineSecu...","{'Contract': 'Month-to-month', 'PaperlessBilli..."
3,0011-IGKFF,Yes,"{'gender': 'Male', 'SeniorCitizen': 1, 'Partne...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'Fiber optic', 'OnlineSecu...","{'Contract': 'Month-to-month', 'PaperlessBilli..."
4,0013-EXCHZ,Yes,"{'gender': 'Female', 'SeniorCitizen': 1, 'Part...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'Fiber optic', 'OnlineSecu...","{'Contract': 'Month-to-month', 'PaperlessBilli..."


# Normaliza a coluna 'customer' para criar um DataFrame separado
 df_customer = pd.json_normalize(df['customer']) 

# #Normaliza a coluna 'customer' e concatena com o DataFrame original
 df = pd.concat([df.drop(columns=['customer']), df_customer], axis=1) 


In [36]:
df.head()  # Exibe as primeiras linhas do DataFrame para verificação

,customerID,Churn,phone,internet,account,gender,SeniorCitizen,Partner,Dependents,tenure
0,0002-ORFBO,No,"{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'DSL', 'OnlineSecurity': '...","{'Contract': 'One year', 'PaperlessBilling': '...",Female,0,Yes,Yes,9
1,0003-MKNFE,No,"{'PhoneService': 'Yes', 'MultipleLines': 'Yes'}","{'InternetService': 'DSL', 'OnlineSecurity': '...","{'Contract': 'Month-to-month', 'PaperlessBilli...",Male,0,No,No,9
2,0004-TLHLJ,Yes,"{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'Fiber optic', 'OnlineSecu...","{'Contract': 'Month-to-month', 'PaperlessBilli...",Male,0,No,No,4
3,0011-IGKFF,Yes,"{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'Fiber optic', 'OnlineSecu...","{'Contract': 'Month-to-month', 'PaperlessBilli...",Male,1,Yes,No,13
4,0013-EXCHZ,Yes,"{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'Fiber optic', 'OnlineSecu...","{'Contract': 'Month-to-month', 'PaperlessBilli...",Female,1,Yes,No,3


# Normaliza a coluna 'phone' para um DataFrame separado(transforma um dicionário[phone] em colunas)
df_phone = pd.json_normalize(df['phone']) 

# Concatena o DataFrame original com as novas coluna normalizado 
df = pd.concat([df.drop(columns=['phone']), df_phone], axis=1) 

df_internet = pd.json_normalize(df['internet'])
df = pd.concat([df.drop(columns=['internet']), df_internet], axis=1)

df_account = pd.json_normalize(df['account'])
df = pd.concat([df.drop(columns=['account']),df_account], axis=1)

df.head()

In [38]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7267 entries, 0 to 7266
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7267 non-null   object 
 1   Churn             7267 non-null   object 
 2   gender            7267 non-null   object 
 3   SeniorCitizen     7267 non-null   int64  
 4   Partner           7267 non-null   object 
 5   Dependents        7267 non-null   object 
 6   tenure            7267 non-null   int64  
 7   PhoneService      7267 non-null   object 
 8   MultipleLines     7267 non-null   object 
 9   InternetService   7267 non-null   object 
 10  OnlineSecurity    7267 non-null   object 
 11  OnlineBackup      7267 non-null   object 
 12  DeviceProtection  7267 non-null   object 
 13  TechSupport       7267 non-null   object 
 14  StreamingTV       7267 non-null   object 
 15  StreamingMovies   7267 non-null   object 
 16  Contract          7267 non-null   object 
